# Etapa 02: Preparação dos dados

### Alvo: binário (0 contra 1–4) ou as 5 classes?

Existem 28 pacientes em `num = 4`. Minha escolha será a classificação binária, pois a divisão entre pacientes doentes e não doentes é relativamente equilibrada.

O problema de manter as 5 classes é que a categoria 4 possui poucos pacientes (aproximadamente 3%), o que dificulta o treinamento. Isso significa que o modelo terá muito menos exemplos para aprender os padrões dessa classe, aumentando o risco de apresentar um desempenho ruim justamente nela.

Além disso, qualquer paciente com `num > 0` já é considerado doente, independentemente da intensidade da doença registrada.

**Decisão:** utilizar classificação binária, agrupando `num = 0` como não doente e `num = 1–4` como doente.

**Por quê:** a classificação binária apresenta uma distribuição mais equilibrada entre as classes e evita o problema de poucas observações na categoria 4.

### Zeros impossíveis: o que vira NaN e por que não substituir diretamente pela média?

Os valores `chol = 0` e `trestbps = 0` serão convertidos para `NaN`, pois representam medidas clinicamente impossíveis ou indicam informações não registradas.

Substituir esses valores pela média imediatamente pode distorcer a distribuição dos dados e introduzir informações artificiais nos registros.

Outro cuidado importante é o data leakage: se a média for calculada utilizando todo o conjunto de dados antes da separação entre treino e teste, informações do conjunto de teste acabarão influenciando o treinamento.

O mais adequado é primeiro converter os zeros para valores nulos, investigar o padrão das ausências e definir uma estratégia de imputação dentro do pipeline de treinamento.

**Decisão:** converter os zeros impossíveis em `NaN`, sem realizar imputação neste momento.

**Por quê:** primeiro é necessário compreender o padrão das ausências e evitar distorções na distribuição dos dados e vazamento de informações.

### Depois da conversão, a Suíça fica com 100% de chol faltando. Isso resolve a armadilha 2 ou só muda ela de lugar?

Na minha percepção, a armadilha apenas muda de lugar.

Anteriormente, `chol = 0` funcionava como um indicador da origem do paciente. Agora, após a conversão, `chol = NaN` pode exercer exatamente o mesmo papel.

Ou seja, o modelo ainda pode aprender que pacientes com colesterol ausente pertencem à Suíça e, consequentemente, associar essa ausência à maior prevalência de doenças observada nesse hospital.

Basicamente, só trocamos o crachá: antes era o zero, agora é o valor ausente.

**Decisão:** não considerar a conversão para `NaN` como uma solução definitiva para o problema.

**Por quê:** o padrão de ausência continua relacionado ao hospital e pode funcionar como um indicador indireto da origem dos pacientes.

### Colunas com muitos nulos (ca 66%, thal 53%, slope 34%): manter ou remover?

Muito provavelmente, eu manteria essas colunas inicialmente, pois, apesar da quantidade elevada de valores ausentes, elas contêm informações clínicas importantes que podem ajudar o modelo a identificar doenças.

**Vantagens:**
- Preservar variáveis clínicas potencialmente relevantes para a previsão.
- Aproveitar as informações disponíveis para identificar padrões associados à doença.
- Evitar descartar características que poderiam contribuir para o desempenho do modelo.

**Desvantagens:**
- A imputação pode introduzir valores artificiais e distorcer a distribuição original.
- Muitas observações precisarão de algum tratamento para os valores ausentes.
- A ausência dessas variáveis está bastante relacionada aos hospitais, então o modelo pode aprender padrões do processo de coleta em vez de características clínicas.

Um ponto que merece atenção especial é a coluna `ca`, pois ela está muito mais preenchida em Cleveland do que nos outros hospitais. Isso significa que a ausência dessa variável pode funcionar como um indicador da origem dos pacientes.

Se a ausência de `ca` estiver fortemente associada aos hospitais, o modelo poderá aprender que `ca = NaN` indica que o paciente não pertence a Cleveland, reproduzindo o mesmo problema de dependência da origem dos dados.

**Decisão:** manter inicialmente as colunas `ca`, `thal` e `slope`, mas investigar cuidadosamente os padrões de ausência antes de definir como tratá-las.

**Por quê:** essas variáveis podem conter informações clínicas importantes, mas seus valores ausentes também podem introduzir vieses relacionados aos hospitais.

### dataset e id: o que fazer com cada uma?

Minha escolha seria remover as colunas `dataset` e `id` das variáveis preditoras, mas mantê-las no DataFrame para análises e validações.

A coluna `dataset` identifica diretamente o hospital de onde o registro veio, enquanto a coluna `id` segue uma sequência de registros organizada por instituição.

Ambas podem funcionar como indicadores indiretos da origem dos pacientes e fazer com que o modelo aprenda padrões que não representam características clínicas da doença.

Além disso, um hospital novo pode não possuir uma categoria conhecida na coluna `dataset`, o que dificulta a generalização do modelo para instituições que não participaram do treinamento.

**Decisão:** remover `dataset` e `id` das variáveis utilizadas pelo modelo, mantendo-as no DataFrame para análises exploratórias e validações.

**Por quê:** essas colunas representam informações relacionadas à origem e à organização dos dados, e não características clínicas que deveriam orientar a previsão.

In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("data/heart_disease_uci.csv")

**Converter chol = 0 e trestbps = 0 em NaN**

      id  age     sex      dataset               cp  trestbps  chol    fbs  \
597  598   32    Male  Switzerland   typical angina      95.0   0.0    NaN   
598  599   34    Male  Switzerland     asymptomatic     115.0   0.0    NaN   
599  600   35    Male  Switzerland     asymptomatic       NaN   0.0    NaN   
600  601   36    Male  Switzerland     asymptomatic     110.0   0.0    NaN   
601  602   38  Female  Switzerland     asymptomatic     105.0   0.0    NaN   
..   ...  ...     ...          ...              ...       ...   ...    ...   
715  716   70    Male  Switzerland     asymptomatic     115.0   0.0  False   
716  717   70    Male  Switzerland     asymptomatic     140.0   0.0   True   
717  718   72    Male  Switzerland      non-anginal     160.0   0.0    NaN   
718  719   73  Female  Switzerland      non-anginal     160.0   0.0  False   
719  720   74    Male  Switzerland  atypical angina     145.0   0.0    NaN   

              restecg  thalch  exang  oldpeak      slope   ca  